# RAG with LangChain- Framework Power

**Module 8 · Lesson 8.5**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- LangChain RAG in 15 Lines - The Express Version
- Document Loaders - One Import per Format
- LCEL - The Pipe Operator and the Runnable Protocol
- Conversational RAG - Memory + Source Citations
- LangGraph - State Machines for Self-Corrective RAG
- Agents and Tools - create_agent, Retriever as a Tool

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## The Rewrite That Took an Afternoon

> 💡 **Info**
>
> A support team shipped the RAG service you built in 8.1, then kept patching it. Six months later it is 900 lines: a bespoke PDF parser, a hand-written retry loop, a homegrown chat-history store, and a chunk of glue that breaks every time someone swaps the embedder. Nobody wants to touch it. A new engineer re-expresses the whole thing in an afternoon: the pipeline becomes about 15 lines of LangChain, the PDF parser becomes one import, the retry loop becomes a LangGraph edge, and - for free - the rewrite gains token streaming, provider fallbacks, and a full trace of every request the original never had. **That is the trade this lesson makes concrete.** You already understand the internals (8.1-8.4). LangChain is how you ship them without hand-building the scaffolding every production system needs.

### 🎯 What you will be able to do after this lesson

- **Build** the 8.1 RAG pipeline in ~15 lines of LangChain (LCEL): loader, splitter, `gemini-embedding-001`, Chroma, and a piped chain into `gemini-2.5-flash`.

- **Compose** with LCEL and the Runnable protocol - the pipe operator, `RunnableParallel`, and why every chain gets streaming, async, and batch for free.

- **Operate** production RAG: self-corrective LangGraph graphs, `create_agent` tool-routing, streaming, caching, fallbacks, and LangSmith traces.

- **Evaluate** the right tool - LCEL vs LangGraph, chain vs agent, and LangChain vs LlamaIndex - by the problem each one actually solves.

> 📦 **Info**
>
> 🧭 Before you start
> You need Lessons 8.1 (the from-scratch pipeline), 8.3 (embeddings and a vector store), and 8.4 (hybrid search: BM25 + dense + RRF - we collapse that into one object here), a Google AI Studio key in `GOOGLE_API_KEY`, and `pip install langchain langchain-google-genai langchain-chroma langchain-text-splitters langgraph langchain-community rank-bm25 chromadb`. Every code block targets **LangChain 1.0** (GA on 2025-10-22): if you copy code from a pre-2025 tutorial, expect dead imports and dead model ids.

## 60-Second Warm-Up: What You Already Know

Three recalls - all load-bearing today. Click a card to check yourself.

> 🔪 **Analogy**
>
> **Your 8.1 build was cooking one dish from raw ingredients - you learned the knife work, the heat, the timing.** LangChain is the professional kitchen: standard stations (swap any component), a defined pass (the LCEL pipe), an expediter who can send a plate back (LangGraph), and cameras on every station (LangSmith). You need both skills. Hand-cooking taught you WHY each step matters; the kitchen lets you run hundreds of covers a night without hand-building the line each time.
> **Where the analogy breaks down:** a kitchen's stations are physical and fixed; LangChain's are software and swappable at runtime - the whole point of the Runnable protocol is that you can replace the embedder or the LLM without touching the chain. And a framework is not faster for ONE dish; the payoff is volume, consistency, and the scaffolding you would never hand-build for a single plate.

> 💡 **Info**
>
> ⚠️ Misconception: "LangChain is just a thin wrapper" (and "LangChain is dead in 2026")
> Two opposite myths, both wrong. It is not a thin wrapper: the value is the **Runnable protocol** (every component speaks one interface, so any station swaps out), the **LangGraph runtime** (loops, branching, persistence a straight chain cannot express), and **observability** (traces the hand-rolled version never had). And it is not dead: LangChain and LangGraph hit **1.0 on 2025-10-22** - a GA reset that made `create_agent` the single agent harness, added middleware, and moved legacy classes to `langchain-classic` (see the [LangChain 1.0 release notes](https://blog.langchain.com/langchain-langgraph-1dot0/) and the [LangGraph repository](https://github.com/langchain-ai/langgraph)). The right mental model is not "less code" as a vanity metric - it is standardized stations, a defined pass, a feedback loop, and cameras.

~60 lines. Full control. No dependencies. **Best for:** learning, custom pipelines, minimal footprint, one-off scripts.

~15 lines. Swappable components. Huge ecosystem + LangGraph + LangSmith. **Best for:** shipping, standard patterns, team projects, production scaffolding.

## Build One: RAG in 15 Lines

Steps 1-4 rebuild your 8.1 pipeline with LangChain: the 15-line express version, the loader ecosystem, LCEL composition itself, and conversational RAG with memory and sources. Every block runs on the current stack.

The LangChain RAG pipeline (same five stages as 8.1)

---

## 🎯 Concept 1: LangChain RAG in 15 Lines - The Express Version

### LangChain RAG in 15 Lines - The Express Version

Load, split, embed, store, retrieve, generate - the 8.1 pipeline, one component each.

**Why this is step 1:** it is the whole lesson on one screen. The pipeline is identical to the 60-line build from 8.1 - load, split, embed, store, retrieve, generate - but every stage is now a LangChain component you connect with a pipe. The "15 lines" number only lands because you already did the 60: you know what each line stands in for.

> 🍽️ **Analogy**
>
> **This is the standard station layout of a professional kitchen.** Prep (loader), portioning (splitter), the sauce base (embeddings), the walk-in (vector store), the line, or brigade (the chain), and the line cook who plates (the LLM). You are not inventing stations; you are assembling the standard ones and sending a ticket down the pass.
> **Where the analogy breaks down:** in a real kitchen you cannot swap the walk-in for a different brand mid-service; here you can - change `Chroma` to `FAISS` or `gemini-embedding-001` to a Hugging Face model, and the rest of the chain does not notice, because every station speaks the same Runnable interface.

You want to swap Chroma for a different vector store. In the 8.1 hand-built version vs the LangChain version, how much of the rest of the pipeline changes?

- **Imports** come from provider packages (`langchain_google_genai`, `langchain_chroma`, `langchain_text_splitters`) - the v1 modular split.

- **Split** turns raw strings into `Document` chunks.

- **Embed + store** is two lines: `gemini-embedding-001` into `Chroma`.

- **The chain** is the interesting line: a dict feeds `context` and `question` into a prompt, then the model, then a parser.

- **Invoke** runs the whole pipeline on one question.

**📝 `01_langchain_rag.py LangChain`** - *1.0*

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install langchain langchain-core langchain-community rank-bm25 langchain-google-genai langchain-text-splitters langchain-chroma chromadb langchain-huggingface sentence-transformers langgraph python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


In [ ]:
# LANGCHAIN RAG - the 8.1 pipeline in ~15 lines
# pip install langchain langchain-google-genai langchain-chroma langchain-text-splitters chromadb
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Documents (in-memory now; step 2 shows real file/PDF/web loaders)
docs = [
    "Netsetos refund policy: full refund within 7 days, 50% from 8-30 days, none after 30.",
    "GenAI course: 14,999 rupees. Lifetime access, Discord, weekly live sessions, certificate.",
    "Live classes daily 7 PM IST on YouTube. Recordings in 2 hours. Python + GCP.",
    "Prerequisites: basic Python and high-school math. No ML experience needed.",
]

# 2. Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.create_documents(docs)

# 3. Embed + store. gemini-embedding-001 (text-embedding-004 was retired 2026-01-14)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 4. LLM. gemini-2.5-flash (gemini-2.0-flash reached end-of-life 2026-06-01)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 5. RAG chain (LCEL)
prompt = ChatPromptTemplate.from_template(
    "Answer using ONLY this context. If it is not there, say you do not know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

# 6. Query
for q in ["What is the refund policy?", "How much does the course cost?"]:
    print(f"Q: {q}\nA: {chain.invoke(q)}\n")

# Output:
# Q: What is the refund policy?
# A: Full refund within 7 days, 50% from 8-30 days, and no refund after 30 days.
# Q: How much does the course cost?
# A: The GenAI course costs 14,999 rupees, with lifetime access included.

#### 💡 What just happened

⚡ What just happened? The same five-stage pipeline from 8.1, now one component per stage: `RecursiveCharacterTextSplitter` (split), `GoogleGenerativeAIEmbeddings` (embed), `Chroma` (store + retrieve), and an **LCEL chain** (`retriever | format | prompt | llm | parse`). The dict literal `{"context": ..., "question": ...}` runs both branches and feeds the prompt. **15 lines replaced 60 - but only because the 8.1 build taught you what each line stands for.** Two facts that make old tutorials fail: `gemini-embedding-001` defaults to 3072 dimensions (the retired 004 was 768), so a straight swap needs a rebuilt store; and `gemini-2.0-flash` now returns an API error.

- Scene 1: a document flows down the pass - loader, splitter, embeddings, vector store, retriever, LLM - one station at a time.

- Scene 2: a query enters, retrieval pulls the matching chunk, and the LLM plates the answer.

- Scene 3: the swap - the embeddings station is replaced with a different model, and the chain keeps running (the Runnable protocol).

- Scene 4: the same output either way - swappable stations, identical contract.

Controls: Play/Pause, Reset, speed (0.5x/1x/2x). The uniform interface is the real product.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Document Loaders - One Import per Format

### Document Loaders - One Import per Format

PDF, HTML, CSV, Notion, Google Drive - each is one import, and each returns the same Document.

**The point of a loader** is not that it reads a PDF - your 8.1 build could do that with a hand-written parser. It is that *every* loader, whatever the format, returns the same `Document` shape (`page_content` + `metadata`), so the rest of your chain never changes when the input format does. Swap a `TextLoader` for a `PyMuPDFLoader` and steps 1's splitter, embedder, and chain are untouched.

> 🫒 **Analogy**
>
> **Loaders are the prep station.** Fish, vegetables, meat - raw ingredients arrive in wildly different forms, and prep turns every one of them into clean, uniform, line-ready mise en place. The sauté station downstream does not care whether the input started as a whole fish or a crate of onions; it receives standardized prep.
> **Where the analogy breaks down:** a prep cook applies judgement per ingredient; a loader is mechanical and lossy - a bad PDF (scanned images, complex tables) still needs a smarter loader (or OCR), and no uniform `Document` shape rescues content the parser never extracted.

You switch your source from a `.txt` file to a PDF. Which parts of the step-1 chain need to change?

- We write a throwaway file so the block actually **runs** (the old version was all comments).

- `TextLoader(path).load()` returns a list of `Document` objects - inspect the type, content, and metadata.

- The catalog shows other loaders; every one of them returns that same `Document` shape.

**📝 `02_loaders.py LangChain`** - *1.0*

In [ ]:
# LANGCHAIN DOCUMENT LOADERS - one import per format, one uniform Document out
# pip install langchain-community pymupdf
import os, tempfile
from langchain_community.document_loaders import TextLoader

# Write a throwaway file so this block actually RUNS
path = os.path.join(tempfile.gettempdir(), "faq.txt")
with open(path, "w", encoding="utf-8") as f:
    f.write("Refund policy: full refund within 7 days.\nSupport: help@netsetos.com")

docs = TextLoader(path).load()          # -> list[Document]
print(type(docs[0]).__name__)           # Document
print(docs[0].page_content[:40])        # Refund policy: full refund within 7 days.
print(docs[0].metadata)                 # the source path, auto-attached

# The catalog: every loader returns the SAME Document shape
catalog = {
    "TextLoader": ".txt files",
    "PyMuPDFLoader": "PDF, one Document per page (+ page metadata)",
    "WebBaseLoader": "any URL -> clean text",
    "CSVLoader": "CSV -> one Document per row",
    "DirectoryLoader": "a whole folder, by glob",
}
for name, desc in catalog.items():
    print(f"{name:16s} -> {desc}")

# Output:
# Document
# Refund policy: full refund within 7 days.
# {'source': '<your-tempdir>/faq.txt'}   # actual temp path varies by OS
# TextLoader       -> .txt files
# PyMuPDFLoader    -> PDF, one Document per page (+ page metadata)
# WebBaseLoader    -> any URL -> clean text
# CSVLoader        -> CSV -> one Document per row
# DirectoryLoader  -> a whole folder, by glob

#### 💡 What just happened

⚡ What just happened? One import, one `.load()`, and you get a list of `Document` objects with `page_content` and `metadata` - regardless of source format. LangChain ships over 160 loaders (PDF, Notion, Confluence, Google Drive, Slack), and they all hand the same shape to the splitter. **That uniformity is the value: your pipeline is written once and fed by anything.** The pre-2025 version of this step was entirely commented out - nothing ran, nothing was verified. **When to use a managed loader vs. a hand-written parser:** reach for the loader for any standard format; its one limitation is a badly-scanned PDF, where you still need OCR or a smarter parser. Shipping an all-commented loader that never runs is the anti-pattern to avoid.

---

## 🎯 Concept 3: LCEL - The Pipe Operator and the Runnable Protocol

### LCEL - The Pipe Operator and the Runnable Protocol

Compose components with `|` like Unix pipes. Every piece is a Runnable.

**Step 1 used the pipe without explaining it.** Here is the machinery. LCEL (LangChain Expression Language) is not clever syntax sugar - the `|` operator builds a `RunnableSequence`, where each step's output is the next step's input. Every component is a *Runnable*: it exposes the same methods (`.invoke`, `.stream`, `.batch`, `.ainvoke`). Compose them any way you like and the whole chain automatically supports streaming, async, and batching - for free.

> 🍴 **Analogy**
>
> **LCEL is the pass in a kitchen.** Each station finishes its part and slides the plate to the next in a fixed order - grill, then sauce, then garnish, then the window. The pass is what makes a brigade composable: every station speaks the same "plate in, plate out" contract, so you can reorder or replace one without redesigning the line.
> **Where the analogy breaks down:** a kitchen pass is strictly one-directional; LCEL can fan OUT too - a dict literal becomes a `RunnableParallel` that runs branches concurrently. But it still cannot loop back ("re-fire this plate") - that is step 5's job (LangGraph).

In `prompt | model | parser`, what is the `|` operator actually doing?

- **Chain 1** is the minimal shape: `prompt | model | parser`.

- **Chain 2** shows a dict becoming a `RunnableParallel` - two prompts run concurrently.

- **Chain 3** is the RAG shape: a retriever branch and a passthrough branch feed one prompt.

- **RunnableLambda** wraps a plain Python function (like `retrieve`) so it becomes a Runnable you can drop into a pipe. LangGraph nodes (step 5) do not need this - the graph wraps them for you.

**📝 `03_lcel_chains.py`** - *LCEL*

In [ ]:
# LCEL - compose Runnables with the | pipe
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# Chain 1: prompt | model | parser  (a RunnableSequence)
explain = ChatPromptTemplate.from_template("Explain {topic} in one sentence.") | llm | StrOutputParser()
print(explain.invoke({"topic": "RAG"}))

# Chain 2: a dict literal becomes RunnableParallel - branches run concurrently
facts = RunnableParallel(
    short=ChatPromptTemplate.from_template("Define {t} in 5 words.") | llm | StrOutputParser(),
    example=ChatPromptTemplate.from_template("Give one real example of {t}.") | llm | StrOutputParser(),
)
print(facts.invoke({"t": "an embedding"}))   # both keys computed in parallel

# Chain 3: RAG shape - retrieve | prompt | llm | parse
def retrieve(q):
    db = {"refund": "Full refund within 7 days.", "price": "14,999 rupees."}
    return next((v for k, v in db.items() if k in q.lower()), "No info found.")

rag = (
    {"context": RunnableLambda(retrieve), "question": RunnablePassthrough()}
    | ChatPromptTemplate.from_template("Context: {context}\nQ: {question}\nAnswer from context:")
    | llm | StrOutputParser()
)
print(rag.invoke("What is the refund policy?"))

# Output:
# RAG injects retrieved documents into an LLM prompt so answers are grounded in your data.
# {'short': 'A vector capturing text meaning', 'example': 'The word king as [0.21, -0.03, ...]'}
# You can get a full refund within 7 days.

#### 💡 What just happened

⚡ What just happened? Three chains, all built with `|`: simple (`prompt | llm | parser`), parallel (a dict of branches run concurrently as a `RunnableParallel`), and RAG (retriever branch + passthrough branch into one prompt). **LCEL is LangChain's composability model: every component is a Runnable, you connect them with pipes, and any chain gains `.stream()`, `.batch()`, and `.ainvoke()` automatically.** Its one ceiling: LCEL is linear - no cycles, no "retry if bad." The moment you need a loop, you reach for LangGraph (step 5). That linearity is LCEL's one limitation, and the tradeoff for its simplicity.

- Scene 1: input flows through `prompt | model | parser` - one Runnable to the next.

- Scene 2: a dict literal forks into a `RunnableParallel` - two branches run at once, then merge.

- Scene 3: tokens stream out of the tail, because every Runnable supports `.stream()`.

- Scene 4: the same chain, called with `.batch()` on many inputs - free, no rewrite.

Controls: Play/Pause, Reset, speed. One protocol, four superpowers.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Conversational RAG - Memory + Source Citations

### Conversational RAG - Memory + Source Citations

Real apps carry chat history and cite where each answer came from.

**A single-shot chain forgets everything between calls.** Production chat needs two things a bare RAG chain lacks: *memory* (so "can I get a refund if I do not like it?" understands what "it" refers to) and *sources* (so the answer says which document it came from). This step shows the explicit, own-it-yourself version - a history list you pass each turn - and names the framework-native path - a checkpointer (LangChain's built-in store that saves conversation state under a `thread_id` so the next call reloads it) - which you meet in steps 5 and 6.

> 🍴 **Analogy**
>
> **Memory is the running order ticket at the pass.** The expediter does not treat each new call as a stranger - the ticket accumulates ("table 7, no onions, allergy noted") so later courses respect earlier context. History is that ticket: each turn appends, and the model reads the whole ticket before answering.
> **Where the analogy breaks down:** a paper ticket grows without limit; a chat history has a token budget - past some length you must summarize or window it, or the context (and cost) balloons. A checkpointer manages that for you; a raw list does not.

Turn 1: "What are the prerequisites?" Turn 2: "How much does it cost?" Turn 3: "Can I get a refund if I do not like it?" What makes turn 3 resolvable?

- Build a small vector store with `metadata` so we can **cite sources**.

- A `ChatPromptTemplate` with a `MessagesPlaceholder` - a named slot the growing `history` list is injected into each turn, so the model sees the whole conversation.

- Loop three turns: retrieve, answer with history + context, then append both messages.

**📝 `04_conversational_rag.py`** - *Production*

In [ ]:
# CONVERSATIONAL RAG - memory + source citations
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

texts = [
    "Netsetos refund: full within 7 days, 50% 8-30 days, none after 30.",
    "GenAI course: 14,999 rupees, lifetime access, Python + GCP.",
    "Prerequisites: basic Python, high-school math. No ML needed.",
]
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
docs = splitter.create_documents(texts, metadatas=[{"source": f"doc_{i}"} for i in range(len(texts))])
vs = Chroma.from_documents(docs, GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"))
retriever = vs.as_retriever(search_kwargs={"k": 2})
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer from context ONLY. If missing, say so.\n\nContext: {context}"),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

history = []          # the running ticket (step 6 uses a checkpointer instead)
for q in ["What are the prerequisites?", "How much does it cost?", "Refund if I do not like it?"]:
    hits = retriever.invoke(q)
    context = "\n".join(d.page_content for d in hits)
    sources = [d.metadata["source"] for d in hits]
    answer = llm.invoke(prompt.format_messages(context=context, history=history, question=q)).content
    history += [HumanMessage(content=q), AIMessage(content=answer)]   # append the turn
    print(f"Q: {q}\nA: {answer.strip()[:90]}\nsources: {sources}\n")

# Output:
# Q: What are the prerequisites?  A: Basic Python and high-school math...  sources: ['doc_2']
# Q: How much does it cost?       A: 14,999 rupees with lifetime access.   sources: ['doc_1']
# Q: Refund if I do not like it?  A: Yes - full refund within 7 days...    sources: ['doc_0']

#### 💡 What just happened

⚡ What just happened? The `MessagesPlaceholder` slots the growing `history` into the prompt each turn, so the model has the thread; `metadata` rides along on every `Document`, so you cite the exact source. **The follow-up "refund if I do not like it?" resolves because prior turns are in the history the model sees.** This manual list is clear and you own it. The framework-native path - `InMemorySaver` (or a Postgres checkpointer) keyed by a `thread_id` - persists that history across sessions and processes without you managing the list; you meet it in step 6.

## Take It to Production

Steps 5-10 are what a framework unlocks that a hand-rolled pipeline rarely gets: self-corrective graphs, agents, streaming and caching, observability, framework retrievers, and the LangChain-vs-LlamaIndex choice. Each carries runnable code, not just bullet points.

---

## 🎯 Concept 5: LangGraph - State Machines for Self-Corrective RAG

### LangGraph - State Machines for Self-Corrective RAG

Nodes, conditional edges, cycles. Grade the retrieval; if it is weak, rewrite and retry.

**LCEL is a straight line; production RAG has loops.** The most common one: retrieve, grade whether the documents are actually relevant, and if not, rewrite the query and retrieve again. You cannot express "go back and try again" in a pipe. LangGraph models your app as a state machine - a `TypedDict` of state, nodes that update it, and edges (some conditional) between them - so a retry becomes an *edge*, not a hack.

> 📣 **Analogy**
>
> **LangGraph is the expediter who can send a plate back.** On a straight pass, whatever reaches the window goes out. An expediter inspects it: undercooked? "Re-fire." Wrong garnish? "Send it back." That feedback loop - inspect, and conditionally route back to an earlier station - is exactly what a state machine adds over a one-way pass.
> **Where the analogy breaks down:** a real expediter uses judgement and can loop forever if the kitchen is failing; a graph needs an explicit *cap* (here, `tries < 2`) or a self-correcting loop can retry indefinitely and burn tokens.

You want "if retrieval is weak, rewrite the query and try again." Can plain LCEL do it?

- `State` is a `TypedDict` - the data every node reads and updates.

- Nodes are plain functions: `retrieve`, `rewrite`, `generate`.

- `grade` is the router - it returns the *name* of the next node as a string, which `add_conditional_edges` maps to the node to run.

- `add_conditional_edges` wires the loop; the `tries` cap stops it.

**📝 `05_langgraph_rag.py`** - *LangGraph*

In [ ]:
# SELF-CORRECTIVE RAG WITH LANGGRAPH - retrieve, grade, rewrite-if-weak, generate
# pip install langgraph langchain-google-genai
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

class State(TypedDict):
    question: str
    query: str
    docs: list
    answer: str
    tries: int

def retrieve(state):                       # swap in your vector store's .invoke()
    corpus = {"refund": "Refund: full within 7 days.", "price": "Course: 14,999 rupees."}
    hits = [v for k, v in corpus.items() if k in state["query"].lower()]
    return {"docs": hits, "tries": state["tries"] + 1}

def grade(state) -> str:                    # the router: relevant docs? or retry?
    if state["docs"]:
        return "generate"
    return "rewrite" if state["tries"] < 2 else "generate"

def rewrite(state):
    better = llm.invoke(f"Rewrite as a short keyword query: {state['question']}").content
    return {"query": better.strip()}

def generate(state):
    ctx = "\n".join(state["docs"]) or "No context found."
    ans = llm.invoke(f"Answer from context only.\nContext: {ctx}\nQ: {state['question']}").content
    return {"answer": ans.strip()}

g = StateGraph(State)
g.add_node("retrieve", retrieve); g.add_node("rewrite", rewrite); g.add_node("generate", generate)
g.add_edge(START, "retrieve")
g.add_conditional_edges("retrieve", grade, {"generate": "generate", "rewrite": "rewrite"})
g.add_edge("rewrite", "retrieve")      # the loop back
g.add_edge("generate", END)
app = g.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "u1"}}
seed = {"question": "How do I get my money back?", "query": "money back",
        "docs": [], "answer": "", "tries": 0}
print(app.invoke(seed, cfg)["answer"])

# Output:
# "money back" misses -> grade routes to rewrite -> a keyword query -> retrieve, then generate
# (once the rewrite matches the corpus) You can get a full refund within 7 days of purchase.

#### 💡 What just happened

⚡ What just happened? The one line that matters is `add_conditional_edges("retrieve", grade, ...)`: after retrieval, `grade` decides whether to `generate` or loop through `rewrite` back to `retrieve`. **That conditional edge IS the difference between a chain and a graph.** The `tries < 2` cap keeps the self-correcting loop from running forever, and `InMemorySaver` checkpoints state so you can resume, inspect, or add human-in-the-loop later. Most production RAG needs at least one such loop (grade retrieval, or validate the answer) - which is why LangGraph, not LCEL, is the production runtime. The tradeoff: a graph is more moving parts than a pipe, so keep plain LCEL when there is no cycle and reach for the graph only when there is.

- Scene 1: the graph - START, retrieve, grade, rewrite, generate, END - laid out as nodes and edges.

- Scene 2: a weak query retrieves nothing; `grade` fires the conditional edge to `rewrite`.

- Scene 3: the rewritten query loops back to `retrieve`, now finds docs, and routes to `generate`.

- Scene 4: the `tries` counter caps the loop so it cannot cycle forever.

Controls: Play/Pause, Reset, speed. The conditional edge is the whole idea.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Agents and Tools - create_agent, Retriever as a Tool

### Agents and Tools - create_agent, Retriever as a Tool

Let the LLM choose the retrieval strategy. One harness: `create_agent`.

**A chain runs a path you wrote; an agent lets the LLM choose the path.** Give the model a set of tools - a vector-store retriever, a web search, a calculator - and it decides which to call for each query, reads the result, and either answers or calls another. In LangChain 1.0 there is exactly one way to build this: `create_agent`. It replaced `initialize_agent`, `AgentExecutor`, AND `create_react_agent`, which all collapsed into it.

> 👀 **Analogy**
>
> **An agent is the head chef routing a ticket.** A ticket comes in; the head chef decides which station it goes to - grill, raw bar, pastry - based on what it is, not a fixed script. Some tickets touch several stations in sequence. The chef routes dynamically; the stations (tools) just do their one job.
> **Where the analogy breaks down:** a head chef is reliable and bounded; an LLM router can loop, call the wrong tool, or never stop - so agents need guardrails (a step cap, middleware, and monitoring) that a human expediter provides by instinct.

You copy an agent tutorial from early 2025 that calls `create_react_agent()`. What happens on LangChain 1.0?

- `@tool` turns a function into a tool; its docstring becomes the description the LLM reads.

- `create_agent` takes a model string, the tools, a system prompt, and a checkpointer.

- Invoke with a `messages` list and a `thread_id`; the agent picks a tool and loops.

**📝 `06_agent_rag.py`** - *create_agent*

In [ ]:
# AGENTIC RAG WITH create_agent - the LLM picks the tool (LangChain 1.0)
# pip install langchain langgraph langchain-google-genai
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_docs(query: str) -> str:
    """Search the Netsetos knowledge base for policies, pricing, and schedules."""
    kb = {"refund": "Full refund within 7 days.", "price": "Course: 14,999 rupees."}
    return next((v for k, v in kb.items() if k in query.lower()), "Not in the knowledge base.")

@tool
def web_search(query: str) -> str:
    """Search the public web for anything not in the knowledge base."""
    return f"[web results for '{query}']"

agent = create_agent(
    model="google_genai:gemini-2.5-flash",        # provider:model string
    tools=[search_docs, web_search],
    system_prompt="Use search_docs for Netsetos questions, web_search otherwise.",
    checkpointer=InMemorySaver(),                # memory keyed by thread_id
)

cfg = {"configurable": {"thread_id": "u1"}}
out = agent.invoke({"messages": [{"role": "user", "content": "What is the refund policy?"}]}, cfg)
print(out["messages"][-1].content)

# Output:
# The agent calls search_docs("refund") -> "Full refund within 7 days." -> final answer:
# You can get a full refund within 7 days of purchase.

#### 💡 What just happened

⚡ What just happened? The LLM read the two tool docstrings, picked `search_docs` for a Netsetos question, ran it, and answered from the result - you never wrote a router. **For real RAG, wrap your retriever with `create_retriever_tool(retriever, "search_docs", "...")` and hand it to `create_agent` the same way.** The critical currency point: `create_agent` is the single LangChain 1.0 harness - `initialize_agent`, `AgentExecutor`, and `create_react_agent` are all deprecated in 1.0 (migrate to create_agent). Customization happens through lightweight hooks the framework runs around the model and tools - the middleware layer - rather than by rewriting the agent class (the deep treatment is Module 11). The tradeoff is control: an agent is flexible but non-deterministic and costs extra model calls, so prefer a fixed chain when the path is known and reach for an agent only when it must be chosen at runtime.

- Scene 1: a query arrives; the agent sees a menu of tools (retriever, web search, calculator).

- Scene 2: the LLM picks one tool based on the query and the tool descriptions.

- Scene 3: the tool returns an observation; the agent reads it.

- Scene 4: the agent either answers or loops to call another tool - the agent loop.

Controls: Play/Pause, Reset, speed. The LLM routes; the tools do one job each.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Production Deployment - Streaming, Caching, Fallbacks

### Production Deployment - Streaming, Caching, Fallbacks

Three guarantees users feel: fast first token, cheap repeats, no hard failures.

**A working chain is not a production service.** Three things separate them, and LCEL gives all three by wrapping - you do not rewrite the chain. *Streaming* (users see tokens as they generate), *caching* (repeat queries skip the model), and *fallbacks* (a provider outage does not crash the app). The same chain from step 1 gains each one with a single call.

> 🍳 **Analogy**
>
> **These are plating for the window.** Streaming is sending each component out the moment it is ready instead of holding the whole plate; caching is the mise that is already prepped so a repeat order goes out instantly; fallbacks are the backup burner when the main one fails mid-service. None of them changes the recipe - they change how reliably and quickly it reaches the guest.
> **Where the analogy breaks down:** a backup burner is identical to the main one; a fallback LLM is a *different* model, so it may answer slightly differently - fallbacks protect availability, not identical output.

To add token streaming to your existing step-1 chain, roughly how much of it do you rewrite?

- **Cache**: `set_llm_cache` once; repeat calls skip the model (`RedisCache` in prod).

- **Fallback**: `.with_fallbacks([...])` fails over to a second model automatically.

- **Stream**: `.astream_events` yields tokens and retriever events as they happen.

**📝 `07_production.py`** - *LCEL*

In [ ]:
# PRODUCTION LCEL - streaming, caching, provider fallbacks (all by wrapping)
import asyncio
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache

set_llm_cache(InMemoryCache())          # repeat calls skip the model (RedisCache in prod)

# Fallback: if the primary errors or rate-limits, fail over automatically
primary = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0, max_retries=0)
backup = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)
llm = primary.with_fallbacks([backup])  # any other chat model works here too

prompt = ChatPromptTemplate.from_template("Answer in one line: {q}")
chain = prompt | llm | StrOutputParser()

# Streaming: same chain, call .astream_events instead of .invoke
async def stream_answer(q):
    async for ev in chain.astream_events({"q": q}, version="v2"):
        if ev["event"] == "on_chat_model_stream":
            print(ev["data"]["chunk"].content, end="", flush=True)

import nest_asyncio; nest_asyncio.apply()  # notebooks already run an event loop
asyncio.run(stream_answer("What is RAG in one line?"))

# Output:
# (tokens arrive one at a time, then form the full sentence)
# RAG retrieves relevant documents and injects them into the prompt for grounded answers.

#### 💡 What just happened

⚡ What just happened? Three production guarantees, zero chain rewrites. `set_llm_cache` makes a repeated query skip the model entirely (`RedisCache` in production; `RedisSemanticCache` even matches near-duplicate phrasings). `.with_fallbacks([backup])` fails over on error or rate limit - set `max_retries=0` on the primary so failover is fast. `.astream_events(version="v2")` yields fine-grained events (tokens, retriever results, tool calls) so a UI can show progress. **To serve it: a few lines of FastAPI that iterate the same async stream. LangServe is in maintenance mode - use FastAPI directly, or LangSmith Deployment (the managed service formerly called LangGraph Platform) for new projects.**

---

## 🎯 Concept 8: LangSmith Observability - Tracing, Callbacks, Evaluation

### LangSmith Observability - Tracing, Callbacks, Evaluation

See exactly what every request did - and catch regressions before users do.

**When a RAG answer is wrong, "why?" has three suspects:** retrieval pulled the wrong docs, the prompt did not ground the model, or the model ignored the context. Without a trace you are guessing. LangSmith records every chain invocation as a tree of runs - query, retrieved documents, formatted prompt, model response - so you can open the exact request and see which suspect is guilty.

> 📷 **Analogy**
>
> **LangSmith is the camera on every station.** A dish comes back to the kitchen - the camera roll shows exactly where it went wrong: the grill overcooked it, or the sauce was skipped, or plating stalled. Without cameras you argue; with them you point. A trace is that footage for one request.
> **Where the analogy breaks down:** a camera is passive; LangSmith also *evaluates* - you can run a golden set through it and score correctness and groundedness automatically, catching a regression before any user sees it.

Your RAG sometimes hallucinates. You enable LangSmith and open a bad trace. What is the single most useful thing to inspect first?

- Two env vars turn on **zero-code tracing** - every `.invoke` is traced.

- A custom `BaseCallbackHandler` logs retrieval and LLM events inline.

- A tiny golden set + a scorer is the seed of `evaluate()` (the full harness is 8.11).

**📝 `08_langsmith.py`** - *LangSmith*

In [ ]:
# LANGSMITH OBSERVABILITY - zero-code tracing + a custom callback + a tiny eval
import os
os.environ["LANGSMITH_TRACING"] = "true"     # plus LANGSMITH_API_KEY and LANGSMITH_PROJECT
# Every chain .invoke() is now traced automatically - no code change.

from langchain_core.callbacks import BaseCallbackHandler

class RetrievalLogger(BaseCallbackHandler):
    def on_retriever_end(self, documents, **kwargs):
        print(f"[trace] retrieved {len(documents)} docs")
    def on_llm_end(self, response, **kwargs):
        print("[trace] llm responded")

# chain.invoke(q, config={"callbacks": [RetrievalLogger()]})

# A tiny offline eval over a golden set (LangSmith's evaluate() scales this)
golden = [
    {"q": "refund window?", "must_contain": "7 days"},
    {"q": "course price?", "must_contain": "14,999"},
]
def groundedness(answer, must):        # 1.0 if the required fact is present
    return 1.0 if must in answer else 0.0

print("golden set:", len(golden), "cases")

# Output:
# golden set: 2 cases
# (with tracing on, each request appears in LangSmith as a run tree you can open)

#### 💡 What just happened

⚡ What just happened? Setting `LANGSMITH_TRACING=true` plus a key gives you full traces with zero code change - a tree of runs showing inputs, outputs, latency, and tokens at every step. A custom `BaseCallbackHandler` hooks `on_retriever_end` and `on_llm_end` for inline metrics. **LangSmith is to LLM apps what DataDog is to web services.** For RAG, the trace localizes a hallucination to retrieval, prompt, or model in seconds; the golden-set `evaluate()` pattern (scored here by hand) is what turns "it felt fine" into a regression gate - the full harness is Lesson 8.11. The tradeoff is small: a little setup and per-trace cost in exchange for pinpointing failures; the alternative is guessing in the dark.

---

## 🎯 Concept 9: Advanced Retrievers - Ensemble, ParentDocument, SelfQuery

### Advanced Retrievers - Ensemble, ParentDocument, SelfQuery

8.4's hand-coded hybrid search, now one object. Plus the retriever families you will reach for.

**Remember the hybrid stack you built by hand in 8.4?** BM25 for exact tokens, dense for meaning, fused with RRF. In LangChain that entire stack is one object: `EnsembleRetriever`. You understand exactly what it does because you built it from scratch - now you get it in a line, plus a family of other retrievers for specific problems.

> 🤝 **Analogy**
>
> **EnsembleRetriever is two cooks working one order.** One cook searches the pantry by exact label (BM25), the other by taste and category (dense); a single expediter merges their picks into one plated result (RRF). You are not choosing one cook - you are combining their complementary strengths, exactly as in 8.4, but now the merging is done for you.
> **Where the analogy breaks down:** two human cooks negotiate; `EnsembleRetriever` merges by a fixed rule (rank fusion, tuned by `weights`), so if one retriever is systematically bad, its weight - not a conversation - is your only lever.

A user searches the exact error code "NTS-4021". Which half of an `EnsembleRetriever` is most likely to surface the right doc?

- Build the **sparse** half: `BM25Retriever.from_texts` (exact tokens).

- Build the **dense** half: a Chroma retriever over `gemini-embedding-001` (meaning).

- `EnsembleRetriever` fuses both with RRF; `weights` tune the blend.

**📝 `09_ensemble_retriever.py`** - *Retrievers*

In [ ]:
# ENSEMBLE RETRIEVER - 8.4's hybrid (BM25 + dense + RRF) as ONE object
# pip install langchain langchain-community rank-bm25 langchain-chroma langchain-google-genai
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

texts = [
    "Refund policy: full refund within 7 days. Error code NTS-4021 means the token expired.",
    "The GenAI course costs 14,999 rupees with lifetime access.",
    "Live classes run daily at 7 PM IST on YouTube.",
]

# Sparse half - exact tokens like NTS-4021
bm25 = BM25Retriever.from_texts(texts); bm25.k = 3

# Dense half - meaning
emb = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
dense = Chroma.from_texts(texts, emb).as_retriever(search_kwargs={"k": 3})

# Fuse with RRF under the hood; weights control the blend
hybrid = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5])

for d in hybrid.invoke("what does NTS-4021 mean"):
    print("-", d.page_content[:52])

# Output:
# - Refund policy: full refund within 7 days. Error code NTS-4021...
# - The GenAI course costs 14,999 rupees with lifetime access.

#### 💡 What just happened

⚡ What just happened?`EnsembleRetriever` IS your 8.4 hybrid search - BM25 plus dense, fused with Reciprocal Rank Fusion - in one object; `weights` is the only knob. Around it sits a family worth knowing: **ParentDocumentRetriever** (index small chunks for precision, return their full parent for context), **SelfQueryRetriever** (the LLM turns "courses under 15,000 rupees" into a metadata filter), **ContextualCompressionRetriever** (trim retrieved docs to the relevant span), and **MultiQueryRetriever** (generate several phrasings, merge). And `SQLRecordManager` makes re-indexing incremental - it hashes content so re-running ingestion skips unchanged docs instead of duplicating them.

- Scene 1: a query hits both retrievers - BM25 produces one ranked list, dense produces another.

- Scene 2: each document gets an RRF score (1/(k+rank)) from each list.

- Scene 3: the weighted scores merge into one fused ranking.

- Scene 4: the exact-code doc, buried by dense, floats to the top because BM25 ranked it first.

Controls: Play/Pause, Reset, speed. This is 8.4's hand-coded stack, now one object.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 10: LangChain vs LlamaIndex + Multilingual RAG

### LangChain vs LlamaIndex + Multilingual RAG

When to reach for each framework - and how to serve Hindi and other languages.

**LangChain is not the only RAG framework, and the choice is not either-or.** LlamaIndex is data-centric (ingestion, indexing, query engines, best-in-class PDF parsing); LangChain is orchestration-centric (agents, tools, state, deployment, LangSmith). The production consensus in 2026 is to use both - LlamaIndex for the data layer, LangChain and LangGraph for orchestration. And whichever you pick, multilingual RAG is mostly an embedding choice.

> 🍝 **Analogy**
>
> **LlamaIndex and LangChain are two kitchens that share a supply chain.** One kitchen specializes in prep and sourcing the best ingredients (LlamaIndex: ingestion, indexing, LlamaParse); the other runs the line and the front of house (LangChain: agents, deployment, observability). A serious restaurant group uses both - it does not force one kitchen to do everything.
> **Where the analogy breaks down:** two kitchens can duplicate work; combining the frameworks adds glue and two dependency trees, so "use both" is a real cost, justified only when each is genuinely better at its layer.

Your core need is ingesting thousands of messy, table-heavy PDFs and querying across them. Which framework leads for that layer?

| Dimension | LangChain | LlamaIndex |
|---|---|---|
| Philosophy | Orchestration-centric | Data / indexing-centric |
| Pure-RAG code | A bit more | Roughly 30-40% less (reported) |
| Agents | LangGraph + create_agent | Workflows (event-driven) |
| Observability | LangSmith (native) | Third-party |
| PDF parsing | Third-party loaders | LlamaParse (best-in-class) |
| GitHub stars (early 2026) | over ~110K | ~45K |

- **Normalize** every string to Unicode NFC first - non-Latin scripts have multiple encodings.

- Use a **multilingual embedder** (`BAAI/bge-m3`) that covers 100+ languages in one model.

- A Hindi query then retrieves the Hindi document - the LLM handles the language, not the framework.

**📝 `10_multilingual.py`** - *Multilingual*

In [ ]:
# MULTILINGUAL RAG - BGE-M3 embeddings handle Hindi, English, and code-switching
# pip install langchain-huggingface sentence-transformers langchain-chroma
import unicodedata
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

def nfc(s):
    return unicodedata.normalize("NFC", s)     # normalize Unicode before indexing

texts = [nfc(t) for t in [
    "रिफंड नीति: 7 दिनों के भीतर पूरा रिफंड।",     # Hindi: full refund within 7 days
    "The GenAI course costs 14,999 rupees.",
]]
emb = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")   # 100+ languages, one model
store = Chroma.from_texts(texts, emb)

for d in store.similarity_search(nfc("पैसे कब वापस मिलेंगे?"), k=1):  # "when do I get money back?"
    print(d.page_content)

# Output:
# रिफंड नीति: 7 दिनों के भीतर पूरा रिफंड।

#### 💡 What just happened

⚡ What just happened? A Hindi query retrieved the Hindi document because `BAAI/bge-m3` embeds 100+ languages into one shared space - the framework did not change, only the embedder. For production Indic RAG, combine it with an `EnsembleRetriever` (BM25 on Hindi-tokenized text + BGE-M3 dense), apply NFC normalization everywhere, and let a checkpointer (the step-6 pattern) carry Hindi/English code-switching - the LLM manages the language, not the framework. **The framework decision and the language decision are separate: pick LlamaIndex or LangChain for architecture, and a multilingual embedder for reach.**

## Putting It Together

You rebuilt the 8.1 pipeline in ~15 lines, then added exactly the scaffolding a hand-rolled system never gets. The through-line: a framework is worth it when its *protocol* (swap any component), its *runtime* (loops and routing), and its *observability* (traces and evals) save you more than the dependency costs you.

> 📦 **Info**
>
> 🔑 Key takeaways
> - **Understand the internals, ship with the framework.** The 8.1 build is why the 15-line version makes sense; do not skip it.
> - **The Runnable protocol is the product.** One interface (`.invoke/.stream/.batch`) means any component swaps out and every chain gets streaming, async, and batch for free.
> - **LCEL for linear, LangGraph for cycles.** The moment you need "retry if weak" or "route by type", that is a graph, not a pipe.
> - **`create_agent` is the one v1.0 harness.** It replaced `initialize_agent`, `AgentExecutor`, and `create_react_agent`; customize with lightweight hooks (middleware), not by rewriting the class.
> - **Production is wrapping, not rewriting.** Streaming, caching, fallbacks, and LangSmith traces attach to a chain you already have.
> - **Frameworks compose.** LlamaIndex for data, LangChain for orchestration; a multilingual embedder is a separate, orthogonal choice.

> ✅ **Info**
>
> 🗺️ Where this goes next
> - **8.6 (RAG with LlamaIndex)** builds this same pipeline the data-centric way - the other kitchen.
> - **8.10 (Agentic RAG)** grows step 5's graph and step 6's agent into a full agentic system with planning and memory.
> - **8.11 (RAG Evaluation)** turns step 8's `evaluate()` into a golden-set + CI harness that gates every change.

*Practice exercises are in the companion practice notebook.*

Eight exercises, easy to hard. Each builds or operates one layer of the LangChain RAG stack.

---

## 🎓 Summary

You've completed the practical part of **RAG with LangChain- Framework Power**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-8_5.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-8.5.html` - regenerate this notebook whenever the source HTML is updated.*
